# Perfumery Type Name Check Research (R3)

**Project:** Carat  
**Check:** `perfumery_type` / requirement R3  
**Purpose:** Evaluate the rules that compare the perfumery type detected in a product name with the value declared in the product-card attribute.

> **Scope.** This notebook evaluates only the rule-based `check_perfumery_name()` implementation from `model/perfumery_type/`. It does not evaluate the ML fallback inside `PerfumeryTypeCheck`, load model weights, or run other Carat checks (GTIN, brand, volume, photos, etc.).

> **Data limitation.** The target implementation description states that some labels in the full dataset may have been generated by the algorithm itself. Therefore, the reported metrics measure agreement with the supplied labels; they must not be presented as an independent estimate of real-world accuracy.

## 1. Environment Setup and Repository Discovery

Run this notebook from Jupyter/JupyterLab using Python 3.12 and an environment that has the repository's algorithm dependencies (`pydantic`) and the research dependencies (`pandas`, `numpy`, `matplotlib`). The notebook intentionally contains no `pip install` commands.

The code searches upward from the current working directory for the Carat repository root, then imports the algorithm package and the research helpers from `model/perfumery_type`.

In [ ]:
from pathlib import Path
import os
import sys
import importlib

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        has_algorithms = (candidate / "algorithms" / "src" / "carat_algorithms").is_dir()
        has_research = (candidate / "model" / "perfumery_type" / "evaluation.py").is_file()
        if has_algorithms and has_research:
            return candidate
    raise FileNotFoundError(
        "Carat repository root not found. Open the notebook from a repository clone "
        "or set the working directory to a folder inside the repository."
    )

REPO_ROOT = find_repo_root(Path.cwd())
ALGORITHMS_SRC = REPO_ROOT / "algorithms" / "src"
RESEARCH_DIR = REPO_ROOT / "model" / "perfumery_type"

for import_path in (str(ALGORITHMS_SRC), str(RESEARCH_DIR)):
    if import_path not in sys.path:
        sys.path.insert(0, import_path)

evaluation = importlib.import_module("evaluation")
perfumery_name_check = importlib.import_module("perfumery_name_check")

STATUSES = list(evaluation.STATUSES)
check_perfumery_name = perfumery_name_check.check_perfumery_name
detect_types_in_name = perfumery_name_check.detect_types_in_name
normalize_declared_type = perfumery_name_check.normalize_declared_type

print("Repository root:", REPO_ROOT)
print("Research module:", RESEARCH_DIR)
print("Statuses:", STATUSES)

## 2. Load the Labeled Dataset

`gold_labeled.csv` is not stored in the repository. Obtain the approved copy from the task owners and set the `PERFUMERY_GOLD_CSV` environment variable before starting Jupyter. If the variable is not set, the notebook looks for `model/perfumery_type/gold_labeled.csv`.

Required columns: `name`, `declared_type`, and `expected_status`. Supported historical labels: `ok`, `warning`, `manual_review`, and `not_checked`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

configured_path = os.environ.get("PERFUMERY_GOLD_CSV")
GOLD_CSV = Path(configured_path).expanduser() if configured_path else (
    REPO_ROOT / "model" / "perfumery_type" / "gold_labeled.csv"
)

if not GOLD_CSV.is_file():
    raise FileNotFoundError(
        f"Labeled dataset not found: {GOLD_CSV}\n"
        "Obtain the approved gold_labeled.csv from the data owner and set "
        "PERFUMERY_GOLD_CSV, or place the file in model/perfumery_type/."
    )

rows = evaluation.load_labeled_csv(GOLD_CSV)
df = pd.read_csv(GOLD_CSV, encoding="utf-8-sig").fillna("")

required_columns = {"name", "declared_type", "expected_status"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required CSV columns: {sorted(missing_columns)}")
if not set(df["expected_status"]).issubset(STATUSES):
    raise ValueError(f"Unknown statuses found in CSV: {set(df['expected_status']) - set(STATUSES)}")
if len(df) != len(rows):
    raise ValueError("The row counts loaded by pandas and csv.DictReader do not match.")

print(f"Dataset file: {GOLD_CSV}")
print(f"Number of rows: {len(df):,}")
display(df.head())

## 3. Dataset Composition

In [ ]:
label_counts = (
    df["expected_status"]
    .value_counts()
    .reindex(STATUSES, fill_value=0)
    .rename_axis("Status")
    .to_frame("Count")
)
label_counts["Share"] = label_counts["Count"] / len(df)
display(label_counts.style.format({"Count": "{:,.0f}", "Share": "{:.2%}"}))

ax = label_counts["Count"].plot(kind="bar", figsize=(8, 4), rot=15)
ax.set_title("Label distribution in the labeled sample")
ax.set_xlabel("Expected status")
ax.set_ylabel("Number of rows")
plt.tight_layout()
plt.show()

## 4. Methods Under Comparison

Three approaches are compared:

1. **Naive first-marker comparison.** Extract the first recognized type from the product name and compare it with the normalized declared attribute.
2. **Conservative rule.** If multiple different types are detected in the name, return `manual_review`.
3. **R3 rule-based implementation.** Use `check_perfumery_name()` from `model/perfumery_type/perfumery_name_check.py`, which delegates to the shared rules in `algorithms`.

The first two approaches are simplified baselines. They use the same parser/type dictionary, so this experiment compares decision rules rather than independent algorithmic systems.

In [ ]:
def naive_first_marker(name: object, declared: object) -> str:
    if not str(name or "").strip() or not str(declared or "").strip():
        return "not_checked"
    detected = detect_types_in_name(name)
    if not detected:
        return "not_checked"
    expected = normalize_declared_type(declared)
    if expected is None:
        return "manual_review"
    return "ok" if detected[0] == expected else "warning"


def conservative_multiple_review(name: object, declared: object) -> str:
    if not str(name or "").strip() or not str(declared or "").strip():
        return "not_checked"
    detected = detect_types_in_name(name)
    if not detected:
        return "not_checked"
    expected = normalize_declared_type(declared)
    if expected is None:
        return "manual_review"
    if len(detected) > 1:
        return "manual_review"
    return "ok" if detected[0] == expected else "warning"


def r3_rule_based(name: object, declared: object) -> str:
    return check_perfumery_name(name, declared)["status"]


y_true = df["expected_status"].tolist()
predictions = {
    "Naive comparison": [
        naive_first_marker(row.name, row.declared_type)
        for row in df.itertuples(index=False)
    ],
    "Conservative rule": [
        conservative_multiple_review(row.name, row.declared_type)
        for row in df.itertuples(index=False)
    ],
    "R3 rule-based": [
        r3_rule_based(row.name, row.declared_type)
        for row in df.itertuples(index=False)
    ],
}
print("Predictions have been generated for all methods.")

## 5. Metric Calculation

In [ ]:
def calculate_metrics(y_expected: list[str], y_predicted: list[str]) -> dict:
    confusion = np.array([
        [sum(expected == label and predicted == column
             for expected, predicted in zip(y_expected, y_predicted))
         for column in STATUSES]
        for label in STATUSES
    ], dtype=int)

    per_class = {}
    f1_values = []
    for index, label in enumerate(STATUSES):
        tp = int(confusion[index, index])
        fp = int(confusion[:, index].sum() - tp)
        fn = int(confusion[index, :].sum() - tp)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        support = int(confusion[index, :].sum())
        per_class[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
        if support:
            f1_values.append(f1)

    correct = int(np.trace(confusion))
    total = len(y_expected)
    return {
        "accuracy": correct / total if total else 0.0,
        "macro_f1": float(np.mean(f1_values)) if f1_values else 0.0,
        "errors": total - correct,
        "confusion_matrix": confusion,
        "per_class": per_class,
    }


metrics = {
    name: calculate_metrics(y_true, y_pred)
    for name, y_pred in predictions.items()
}

# Cross-check R3 against the official evaluator in model/perfumery_type/evaluation.py.
official_report = evaluation.evaluate(rows)
r3_metrics = metrics["R3 rule-based"]
assert round(r3_metrics["accuracy"], 4) == official_report["accuracy"]
assert round(r3_metrics["macro_f1"], 4) == official_report["macro_f1"]
assert r3_metrics["errors"] == len(official_report["mistakes"])

print("R3 metrics match the official evaluation.py output.")

## 6. Method Comparison

In [ ]:
summary = pd.DataFrame([
    {
        "Method": name,
        "Accuracy": result["accuracy"],
        "Macro-F1": result["macro_f1"],
        "Mismatches": result["errors"],
    }
    for name, result in metrics.items()
]).set_index("Method")

display(summary.style.format({
    "Accuracy": "{:.2%}",
    "Macro-F1": "{:.2%}",
    "Mismatches": "{:,.0f}",
}))

In [ ]:
ax = summary[["Accuracy", "Macro-F1"]].plot(
    kind="bar", figsize=(9, 5), ylim=(0, 1.05), rot=12
)
ax.set_title("Method comparison: Accuracy and Macro-F1")
ax.set_xlabel("")
ax.set_ylabel("Metric value")
plt.tight_layout()
plt.show()

ax = summary["Mismatches"].plot(kind="bar", figsize=(8, 4), rot=12)
ax.set_title("Mismatches against the supplied labels")
ax.set_xlabel("")
ax.set_ylabel("Number of rows")
plt.tight_layout()
plt.show()

## 7. R3 Metrics by Class

In [ ]:
class_table = pd.DataFrame([
    {
        "Status": label,
        "Precision": values["precision"],
        "Recall": values["recall"],
        "F1-score": values["f1"],
        "Support": values["support"],
    }
    for label, values in r3_metrics["per_class"].items()
]).set_index("Status")

display(class_table.style.format({
    "Precision": "{:.2%}",
    "Recall": "{:.2%}",
    "F1-score": "{:.2%}",
    "Support": "{:,.0f}",
}))

## 8. R3 Confusion Matrix

In [ ]:
cm = r3_metrics["confusion_matrix"]
fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(cm, interpolation="nearest")
ax.set_title("R3 rule-based confusion matrix")
ax.set_xticks(range(len(STATUSES)))
ax.set_yticks(range(len(STATUSES)))
ax.set_xticklabels(STATUSES, rotation=20, ha="right")
ax.set_yticklabels(STATUSES)
ax.set_xlabel("Predicted status")
ax.set_ylabel("Expected status")

threshold = cm.max() / 2 if cm.size else 0
for i in range(len(STATUSES)):
    for j in range(len(STATUSES)):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center", va="center",
            color="white" if cm[i, j] > threshold else "black",
        )
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

## 9. Mismatch Analysis

For R3, the notebook uses the official mismatch list returned by `evaluation.evaluate()`. Each mismatch also includes the rule identifier that produced the result.

In [ ]:
mistakes = pd.DataFrame(official_report["mistakes"])
print(f"Number of R3 mismatches: {len(mistakes)}")
if not mistakes.empty:
    display(mistakes)
else:
    print("No mismatches found.")

bundle_mistakes = mistakes[mistakes["rule"] == "PERFUMERY_BUNDLE_SKIPPED"] if not mistakes.empty else mistakes
print(f"Mismatches related to bundles: {len(bundle_mistakes)}")
if not bundle_mistakes.empty:
    display(bundle_mistakes[["name", "declared_type", "expected", "predicted", "rule"]])

## 10. Interpretation and Limitations

On the currently supplied 15,000-row dataset, the R3 rule-based implementation agrees with the provided labels on 14,993 rows. All seven mismatches involve the `КОМПЛЕКТ` (bundle) attribute: the `PERFUMERY_BUNDLE_SKIPPED` rule returns `not_checked`, while the labels contain `ok` for six rows and `manual_review` for one row.

The result must not be interpreted as an independent quality estimate:

- The target implementation description states that some labels in the full dataset may have been generated by the algorithm itself.
- The `manual_review` class contains only four examples.
- The baseline methods use the same parser and type dictionary as R3.
- This notebook evaluates only the rule-based function; it does not run the ML fallback inside `PerfumeryTypeCheck` or load model weights.

An independent evaluation requires a separate expert-labeled sample created without using the algorithm under evaluation. The expected handling of product bundles should also be agreed with the owners of the requirements and labels.